In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)
abandono

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_curve, roc_auc_score, auc, average_precision_score, precision_recall_curve
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import RocCurveDisplay

## **Abandono**

In [ ]:
def weekly_indicadores_clf(w, ax, indicadores):
  w_ind = indicadores.copy(deep=True)
  w_ind['passed'] = abandono['curso_superado']
  w_ind['dropped_out'] = abandono['abandono']

  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
  w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
  w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

  X = w_ind.drop(['dropped_out','passed'],axis=1)
  y = w_ind['dropped_out']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  clf = RandomForestClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
  print('\tscore:',np.round(clf.score(X_train, y_train),4))
  print('\ttest score:',np.round(clf.score(X_test,y_test),4))

  y_pred = clf.predict_proba(X_test)[:,1]
  auc = roc_auc_score(y_test, y_pred)
  mse = mean_squared_error(y_test, y_pred)
  rmse = mse**.5

  roc = RocCurveDisplay.from_estimator(clf, X_test, y_test, ax=ax, alpha=0.8, name='Week {}'.format(w))

  return auc, rmse, roc

In [ ]:
aucs = {}
rmses = {}
for w in range(1,8):
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  print('\nWeek', w)
  auc, rmse, roc =  weekly_indicadores_clf(w, ax = plt.gca(), indicadores = ind)
  aucs['Week {}'.format(str(w))] = auc
  rmses['Week {}'.format(str(w))] = rmse
  print('\tauc score:', np.round(auc,4))
  print('\trmse score:', np.round(rmse,4))

In [ ]:
fig = plt.subplots(1,2, figsize=(14,7))

x, y = zip(*aucs.items())
plt.subplot(1, 2, 1)
plt.plot(x, y, marker='o')
plt.title('AUC')

x, y = zip(*rmses.items())
plt.subplot(1, 2, 2)
plt.plot(x, y, marker='o')
plt.title('RMSE')
plt.show()

## **Nota final**


In [ ]:
def weekly_indicadores_reg(w, indicadores):
  w_ind = indicadores.copy(deep=True)
  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)

  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  X = w_ind.drop(['grade'],axis=1)
  y = w_ind['grade']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  reg = RandomForestRegressor(max_depth=2, random_state=42).fit(X_train, y_train)
  print('\tscore:',np.round(reg.score(X_train, y_train),4))
  print('\ttest score:',np.round(reg.score(X_test,y_test),4))

  y_pred = reg.predict(X_test)
  mse = mean_squared_error(y_test, y_pred)
  rmse = mse**.5

  return rmse

In [ ]:
rmses = {}
for w in range(1,8):
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  print('\nWeek', w)
  rmse = weekly_indicadores_reg(w,ind)
  rmses['Week {}'.format(str(w))] = rmse
  print('\trmse score:', np.round(rmse,4))

In [ ]:
x, y = zip(*rmses.items())
plt.plot(x, y, marker='o')
plt.title('RMSE')
plt.show()